In [25]:
import os
import gc
from pathlib import Path
import time
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction import _stop_words
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder
import cohere
import faiss
from rank_bm25 import BM25Okapi

load_dotenv(Path("..") / ".env")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

cohere_api_key = os.getenv('COHERE_API_KEY')
co = cohere.ClientV2(cohere_api_key)

encoder = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('ms-marco-MiniLM-L6-v2')
qdrant = QdrantClient(":memory:")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

d:\Repos\RAG\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\meank\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Chunking -     Automatically done by using notes attribute - notes reflect details about a single coherent unit ie. a particular wine
Encoder -            Sentence Transformer | Cohere API
Vector Store - Qdrant In-memory vector db | Faiss Index
Search -       DB/Index Specific

In [3]:
# Read data
df = pd.read_csv('../data/top_rated_wines.csv')
df = df[df['variety'].notna()] # remove any NaN values as it blows up serialization
data = df.to_dict('records')
df.head()

,name,region,variety,rating,notes
0,3 Rings Reserve Shiraz 2004,"Barossa Valley, Barossa, South Australia, Aust...",Red Wine,96.0,Vintage Comments : Classic Barossa vintage con...
1,Abreu Vineyards Cappella 2007,"Napa Valley, California",Red Wine,96.0,Cappella is a proprietary blend of two clones ...
2,Abreu Vineyards Cappella 2010,"Napa Valley, California",Red Wine,98.0,Cappella is one of the oldest vineyard sites i...
3,Abreu Vineyards Howell Mountain 2008,"Howell Mountain, Napa Valley, California",Red Wine,96.0,When David purchased this Howell Mountain prop...
4,Abreu Vineyards Howell Mountain 2009,"Howell Mountain, Napa Valley, California",Red Wine,98.0,"As a set of wines, it is hard to surpass the f..."


In [4]:
# Create Chunks
texts = [doc["notes"] for doc in data]
text_ids = [idx for idx,_ in enumerate(data)]
print(len(texts))

1347


In [26]:
# Create Embeddings
def embed_texts(texts, method, batch_size=96):
    if method=="cohere":
        embeddings=[]
        for i in range(0, len(texts), batch_size): # Running in batches bcoz Cohere trial API starts hittinh rate limit of 100k tokens/min
            batch = texts[i:i+batch_size]
            embeds = co.embed(
                texts=batch,
                input_type="search_document",
                model="embed-v4.0",
                embedding_types=["float"]
                ).embeddings.float # Since .embeddings return a .float list (EmbedTypeResponseEmbeddings object) and not a raw list of vectors
            embeddings.extend(embeds) # extend and not append since we are running in batches. Otherwise it will store each 96 embeddings in another sub-list indexed by batch_no
            time.sleep(60)
    elif method=="sentence_transformer":
        embeddings = encoder.encode(texts)
    else:
        raise ValueError(f"Invalid method: {method}")
    embeddings = np.array(embeddings)
    return embeddings

In [27]:
# Create VectorDB
def create_vector_db(db_type, doc_embeds, qdrant_db_name='default'):
    if db_type=='qdrant':
        qdrant.create_collection(
            collection_name=qdrant_db_name,
            vectors_config=models.VectorParams(
                size=doc_embeds.shape[1],
                distance=models.Distance.COSINE
            )
        )
        qdrant.upload_points(
            collection_name=qdrant_db_name,
            points=[
                models.PointStruct(
                    id=idx,
                    vector=doc_embeds[idx].tolist(),
                    #payload=doc,
                ) for idx in range(doc_embeds.shape[0])
            ]
        )
        return None
    elif db_type=='faiss':
        dim = doc_embeds.shape[1]
        index = faiss.IndexFlatL2(dim)
        print(index.is_trained)
        index.add(np.float32(doc_embeds))
        return index

In [28]:
def search(query, method, vector_db, db_type, qdrant_db_name='default', num_results=3):
    if method=="cohere":
        query_embed = co.embed(
            texts=[query],
            input_type="search_query",
            model="embed-v4.0",
            embedding_types=['float']
            ).embeddings.float[0] # .embeddings.float returns a list of vectors
    elif method=="sentence_transformer":
        query_embed = encoder.encode(query).tolist()
    
    if db_type=='faiss':
        scores, ids = vector_db.search( # scores → distances, e.g. [[0.12, 0.34, 0.41]], ids → neighbor indices, e.g. [[42, 17, 9]]
            np.float32(query_embed).reshape(1,-1), # FAISS expects a 2D array, query_embed = 1D array
            num_results
        )
        search_results = []
        for id, score in zip(ids[0], scores[0]):
            result = {"doc": data[id], "score": score}
            search_results.append(result)
        return search_results
        
    elif db_type=='qdrant':
        hits = qdrant.query_points(
            collection_name = qdrant_db_name,
            query = query_embed,
            limit=num_results
        )
        search_results = []
        for hit in hits.points:
            result = {"doc": data[hit.id], "score": hit.score}
            search_results.append(result)
    
        return search_results

In [8]:
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = re.sub(r'[^\w\s]', '', token) # removing anything which is not a word or whitespace
        token = token.strip()

        if len(token)>0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc 

In [9]:
def bm25_search(query, model, num_results=3):
    bm25_scores = model.get_scores(bm25_tokenizer(query))
    hits_indexes = np.argpartition(bm25_scores, -num_results)[-num_results:]
    hits = [{'id':idx, 'score':bm25_scores[idx]} for idx in hits_indexes]
    hits = sorted(hits, key=lambda x: x['score'], reverse=True)
    
    search_results = []
    for hit in hits:
        result = {'doc': data[hit['id']], 'score': hit['score']}
        search_results.append(result)
    
    return search_results

In [31]:
def rerank(query, method, candidates, top_k=3):
    if method=='cohere':
        results = co.rerank(model="rerank-v4.0-pro",query=query, documents=candidates, top_n=top_k)
        return results.results
    elif method=='cross_encoder':
        cross_inp = [[query, candidate] for candidate in candidates]
        cross_scores = reranker.predict(cross_inp)
        results = [{'score':cross_score, 'candidate':candidate} for cross_score, candidate in zip(cross_scores,candidates)]
        results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
        return results

# Sparse Retrieval

In [23]:
bm25_corpus = []
for text in texts:
    bm25_corpus.append(bm25_tokenizer(text))

bm25_model = BM25Okapi(bm25_corpus)

query = "Suggest me an amazing Malbec wine from Argentina"

search_results = bm25_search(query, bm25_model, num_results=15)
for result in search_results:
    print("Score:", result["score"], "Answer: ", result["doc"]["notes"])

Score: 11.086648372867462 Answer:  "The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbec to the 

# Dense Retrieval

## Embeddings = Cohere
## Vector DB = FAISS

In [11]:
query = "Suggest me an amazing Malbec wine from Argentina"

document_embeddings = embed_texts(texts, method='cohere')
print(document_embeddings.shape)

index = create_vector_db('faiss', document_embeddings)

search_results = search(query,method='cohere',vector_db=index,db_type='faiss', num_results=15)
for result in search_results:
    print("Score:", result["score"], "Answer: ", result["doc"]["notes"])

(1347, 1536)
True
Score: 0.8375702 Answer:  "The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbe

## Embeddings = Cohere
## Vector DB = Qdrant

In [10]:
query = "Suggest me an amazing Malbec wine from Argentina"

#document_embeddings = embed_texts(texts, method='cohere')
print(document_embeddings.shape)

index = create_vector_db('qdrant', document_embeddings)

search_results = search(query,method='cohere',vector_db=None,db_type='qdrant')
for result in search_results:
    print("Score:", result["score"], "Answer: ", result["doc"]["notes"])

(1347, 1536)
Score: 0.5812149821493925 Answer:  "The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting M

## Embeddings = Sentence Transformer
## Vector DB = FAISS

In [30]:
query = "Suggest me an amazing Malbec wine from Argentina"

document_embeddings = embed_texts(texts, method='sentence_transformer')
print(document_embeddings.shape)

index = create_vector_db('faiss', document_embeddings)

search_results = search(query,method='sentence_transformer',vector_db=index,db_type='faiss', num_results=15)
for result in search_results:
    print("Score:", result["score"], "Answer: ", result["doc"]["notes"])

(1347, 384)
True
Score: 0.7244436 Answer:  "The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbec

# ReRank

## Embeddings = Cohere
## Vector DB = FAISS
## ReRank = Cohere

In [ ]:
candidates = [result["doc"]["notes"] for result in search_results]

rerank_results = rerank(query, method='cohere', candidates, top_k=3)

for result in rerank_results:
    print(result.relevance_score, candidates[result.index])

0.9389352 "The single-vineyard 2004 Malbec Nicasia Vineyard is located in the Altamira district of Mendoza. It was aged for 18 months in new French oak. Opaque purple-colored, it exhibits a complex perfume of pain grille, scorched earth, mineral, licorice, blueberry, and black cherry. Thick on the palate, bordering on opulent, it has layers of fruit, silky tannins, and a long, fruit-filled finish. It will age effortlessly for another 6-8 years and provide pleasure through 2025. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbec to the United States in 1

## Embeddings = Sentence Transformer
## Vector DB = FAISS
## ReRank = Cross Encoder

In [35]:
candidates = [result["doc"]["notes"] for result in search_results]

rerank_results = rerank(query, 'cross_encoder', candidates, top_k=3)

for result in rerank_results:
    print(result['score'], result['candidate'])

4.961233 "The single-vineyard 2004 Malbec Argentino Vineyard spent 17 months in new French oak. Remarkably fragrant and complex aromatically, it offers up aromas of wood smoke, creosote, pepper, clove, black cherry, and blackberry. Made in a similar, elegant style, it is the most structured of the three single vineyard wines, needing a minimum of a decade of additional cellaring. It should easily prove to be a 25-40 year wine. It is an exceptional achievement in Malbec. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbec to the United States in 1994."
4.

## Embeddings = Tokens
## Vector DB = BM25
## ReRank = Cohere

In [24]:
candidates = [result["doc"]["notes"] for result in search_results]

rerank_results = rerank(query, candidates, top_k=3)

for result in rerank_results:
    print(result.relevance_score, candidates[result.index])

0.9388232 "The single-vineyard 2004 Malbec Nicasia Vineyard is located in the Altamira district of Mendoza. It was aged for 18 months in new French oak. Opaque purple-colored, it exhibits a complex perfume of pain grille, scorched earth, mineral, licorice, blueberry, and black cherry. Thick on the palate, bordering on opulent, it has layers of fruit, silky tannins, and a long, fruit-filled finish. It will age effortlessly for another 6-8 years and provide pleasure through 2025. When all is said and done, Catena Zapata is the Argentina winery of reference – the standard of excellence for comparing all others. The brilliant, forward-thinking Nicolas Catena remains in charge, with his daughter, Laura, playing an increasingly large role. The Catena Zapata winery is an essential destination for fans of both architecture and wine in Mendoza. It is hard to believe, given the surge in popularity of Malbec in recent years, that Catena Zapata only began exporting Malbec to the United States in 1

In [36]:
del qdrant
gc.collect()

4202